In [25]:
from herbie import Herbie
import cfgrib
import pandas as pd
import numpy as np
import xarray as xr
import pathlib, shutil
 
RUN     = pd.Timestamp("2025-04-29 00:00")
LAT_PSU = 40.79
LON_PSU = -77.86
FXX     = range(0, 97, 3)
 
SAVE_DIR = pathlib.Path("data")
SAVE_DIR.mkdir(exist_ok=True)
NC_PATH  = SAVE_DIR / "ifs_psu_apr29_2025.nc"
 
# ── Helper: find a specific variable across cfgrib datasets ──────────────────
def get_var(datasets, varname):
    """
    Search all cfgrib datasets in a file for a variable by name.
    Returns the DataArray for that variable.
    """
    for ds in datasets:
        if varname in ds.data_vars:
            return ds[varname]
    raise KeyError(f"'{varname}' not found. Available: "
                   + str([v for ds in datasets for v in ds.data_vars]))
 
def extract_point(da):
    """Extract nearest grid point to Penn State."""
    return float(da.sel(latitude=LAT_PSU, longitude=LON_PSU,
                        method="nearest").values)
 
# ── Download loop ─────────────────────────────────────────────────────────────
print("Starting download loop…")
records = []
 
for f in FXX:
    try:
        H          = Herbie(RUN, model="ifs", product="oper", fxx=f)
        grib_path  = H.download()                        # full GRIB2, no index
        datasets   = cfgrib.open_datasets(grib_path)     # all messages
 
        tp_raw = extract_point(get_var(datasets, "tp"))    # total precip (m)
        gust   = extract_point(get_var(datasets, "fg10"))  # 10-m gust (m/s)
        t_k    = extract_point(get_var(datasets, "t2m"))   # 2-m temp (K)
        d_k    = extract_point(get_var(datasets, "d2m"))   # 2-m dewpoint (K)
 
        records.append({"fxx": f, "tp_m": tp_raw,
                        "gust": gust, "t_k": t_k, "d_k": d_k})
        print(f"  F{f:03d}  T={t_k-273.15:.1f}°C ({(t_k-273.15)*9/5+32:.1f}°F)"
              f"  gust={gust*2.237:.1f} mph  tp={tp_raw*1000:.3f} mm")
 
    except Exception as e:
        print(f"  F{f:03d} FAILED: {e}")
        records.append({"fxx": f, "tp_m": np.nan, "gust": np.nan,
                        "t_k": np.nan, "d_k": np.nan})
 
# ── Save to NetCDF ────────────────────────────────────────────────────────────
df_raw = pd.DataFrame(records)
ds_out = xr.Dataset.from_dataframe(df_raw.set_index("fxx"))
ds_out.to_netcdf(NC_PATH)
print(f"\nSaved → {NC_PATH}")
 
# ── Clean up downloaded GRIB files ────────────────────────────────────────────
herbie_cache = pathlib.Path("~/data").expanduser()
if herbie_cache.exists():
    shutil.rmtree(herbie_cache, ignore_errors=True)
    print("Temporary GRIB files removed.")
 
# ── Unit conversions ──────────────────────────────────────────────────────────
df = df_raw.sort_values("fxx").reset_index(drop=True)
 
df["tp_mm"]    = np.diff(np.insert(df["tp_m"].values * 1000, 0, 0))
df["temp_c"]   = df["t_k"]  - 273.15
df["temp_f"]   = df["temp_c"] * 9/5 + 32
df["dew_c"]    = df["d_k"]  - 273.15
df["gust_mph"] = df["gust"] * 2.237
 
print("\n=== Sanity check (first 8 rows) ===")
print(df[["fxx","temp_f","gust_mph","tp_mm"]].head(8).to_string())

Starting download loop…
✅ Found ┊ model=ifs ┊ product=oper ┊ 2025-Apr-29 00:00 UTC F00 ┊ GRIB2 @ google ┊ IDX @ google


Ignoring index file '/home/fmf5170/data/ifs/20250429/20250429000000-0h-oper-fc.grib2.5b7b6.idx' incompatible with GRIB file
Ignoring index file '/home/fmf5170/data/ifs/20250429/20250429000000-0h-oper-fc.grib2.5b7b6.idx' incompatible with GRIB file
Ignoring index file '/home/fmf5170/data/ifs/20250429/20250429000000-0h-oper-fc.grib2.5b7b6.idx' incompatible with GRIB file
Ignoring index file '/home/fmf5170/data/ifs/20250429/20250429000000-0h-oper-fc.grib2.5b7b6.idx' incompatible with GRIB file
Ignoring index file '/home/fmf5170/data/ifs/20250429/20250429000000-0h-oper-fc.grib2.5b7b6.idx' incompatible with GRIB file
Ignoring index file '/home/fmf5170/data/ifs/20250429/20250429000000-0h-oper-fc.grib2.5b7b6.idx' incompatible with GRIB file
Ignoring index file '/home/fmf5170/data/ifs/20250429/20250429000000-0h-oper-fc.grib2.5b7b6.idx' incompatible with GRIB file
Ignoring index file '/home/fmf5170/data/ifs/20250429/20250429000000-0h-oper-fc.grib2.5b7b6.idx' incompatible with GRIB file
Ignoring

  F000  T=16.7°C (62.1°F)  gust=0.0 mph  tp=0.000 mm
✅ Found ┊ model=ifs ┊ product=oper ┊ 2025-Apr-29 00:00 UTC F03 ┊ GRIB2 @ google ┊ IDX @ google


Ignoring index file '/home/fmf5170/data/ifs/20250429/20250429000000-3h-oper-fc.grib2.5b7b6.idx' incompatible with GRIB file
Ignoring index file '/home/fmf5170/data/ifs/20250429/20250429000000-3h-oper-fc.grib2.5b7b6.idx' incompatible with GRIB file
Ignoring index file '/home/fmf5170/data/ifs/20250429/20250429000000-3h-oper-fc.grib2.5b7b6.idx' incompatible with GRIB file
Ignoring index file '/home/fmf5170/data/ifs/20250429/20250429000000-3h-oper-fc.grib2.5b7b6.idx' incompatible with GRIB file
Ignoring index file '/home/fmf5170/data/ifs/20250429/20250429000000-3h-oper-fc.grib2.5b7b6.idx' incompatible with GRIB file
Ignoring index file '/home/fmf5170/data/ifs/20250429/20250429000000-3h-oper-fc.grib2.5b7b6.idx' incompatible with GRIB file
Ignoring index file '/home/fmf5170/data/ifs/20250429/20250429000000-3h-oper-fc.grib2.5b7b6.idx' incompatible with GRIB file
Ignoring index file '/home/fmf5170/data/ifs/20250429/20250429000000-3h-oper-fc.grib2.5b7b6.idx' incompatible with GRIB file
Ignoring

  F003  T=14.5°C (58.1°F)  gust=20.4 mph  tp=0.000 mm
✅ Found ┊ model=ifs ┊ product=oper ┊ 2025-Apr-29 00:00 UTC F06 ┊ GRIB2 @ google ┊ IDX @ google


...

  F093 FAILED: "'fg10' not found. Available: ['tcw', 'tcwv', 'u100', 'v100', 'u10', 'v10', 'fg10_3', 't2m', 'd2m', 'mx2t3', 'mn2t3', 't', 'u', 'v', 'q', 'w', 'vo', 'd', 'gh', 'r', 'msl', 'mucape', 'ttr', 'vsw', 'sot', 'asn', 'sp', 'ssrd', 'lsm', 'strd', 'ssr', 'str', 'ewss', 'nsss', 'ro', 'tp', 'skt', 'ptype', 'tprate', 'sithick', 'zos', 'svn', 'sve']"
✅ Found ┊ model=ifs ┊ product=oper ┊ 2025-Apr-29 00:00 UTC F96 ┊ GRIB2 @ google ┊ IDX @ google


Ignoring index file '/home/fmf5170/data/ifs/20250429/20250429000000-96h-oper-fc.grib2.5b7b6.idx' incompatible with GRIB file
Ignoring index file '/home/fmf5170/data/ifs/20250429/20250429000000-96h-oper-fc.grib2.5b7b6.idx' incompatible with GRIB file
Ignoring index file '/home/fmf5170/data/ifs/20250429/20250429000000-96h-oper-fc.grib2.5b7b6.idx' incompatible with GRIB file
Ignoring index file '/home/fmf5170/data/ifs/20250429/20250429000000-96h-oper-fc.grib2.5b7b6.idx' incompatible with GRIB file
Ignoring index file '/home/fmf5170/data/ifs/20250429/20250429000000-96h-oper-fc.grib2.5b7b6.idx' incompatible with GRIB file
Ignoring index file '/home/fmf5170/data/ifs/20250429/20250429000000-96h-oper-fc.grib2.5b7b6.idx' incompatible with GRIB file
Ignoring index file '/home/fmf5170/data/ifs/20250429/20250429000000-96h-oper-fc.grib2.5b7b6.idx' incompatible with GRIB file
Ignoring index file '/home/fmf5170/data/ifs/20250429/20250429000000-96h-oper-fc.grib2.5b7b6.idx' incompatible with GRIB file


  F096 FAILED: "'fg10' not found. Available: ['tcw', 'tcwv', 'u100', 'v100', 'u10', 'v10', 'fg10_3', 't2m', 'd2m', 'mx2t3', 'mn2t3', 't', 'u', 'v', 'q', 'w', 'vo', 'd', 'gh', 'r', 'msl', 'mucape', 'ttr', 'vsw', 'sot', 'asn', 'sp', 'ssrd', 'lsm', 'strd', 'ssr', 'str', 'ewss', 'nsss', 'ro', 'tp', 'skt', 'ptype', 'tprate', 'sithick', 'zos', 'svn', 'sve']"

Saved → data/ifs_psu_apr29_2025.nc


sh: 1: getfattr: not found


Temporary GRIB files removed.

=== Sanity check (first 8 rows) ===
   fxx     temp_f   gust_mph     tp_mm
0    0  62.105220   0.000000  0.000000
1    3  58.107405  20.447038  0.000000
2    6  54.529268  20.625119  0.000000
3    9  52.157869  17.765498  0.000000
4   12  56.250331  17.620356  0.000000
5   15  71.958748  23.377738  0.000000
6   18  79.227522  32.100208  0.568390
7   21  73.486342  19.394579  4.169464
